In [ ]:
# EMSXHistoryExample_Final.py
import blpapi
import sys
import time

# 定义事件和消息类型常量
SESSION_STARTED = blpapi.Name("SessionStarted")
SESSION_STARTUP_FAILURE = blpapi.Name("SessionStartupFailure")
SERVICE_OPENED = blpapi.Name("ServiceOpened")
SERVICE_OPEN_FAILURE = blpapi.Name("ServiceOpenFailure")
ERROR_INFO = blpapi.Name("ErrorInfo")
GET_FILLS_RESPONSE = blpapi.Name("GetFillsResponse")

# 服务配置：按需切换（优先测试uat，再验证生产）
# 生产环境
# HISTORY_SERVICE = "//blp/emsx.history"
# UAT测试环境（优先尝试）
HISTORY_SERVICE = "//blp/emsx.history.uat"
HOST = "localhost"
PORT = 8194
b_end = False  # 控制程序退出的标志


class SessionEventHandler:
    def processEvent(self, event, session):
        """处理所有类型的事件"""
        try:
            if event.eventType() == blpapi.Event.SESSION_STATUS:
                self.process_session_status(event, session)
            elif event.eventType() == blpapi.Event.SERVICE_STATUS:
                self.process_service_status(event, session)
            elif event.eventType() in (blpapi.Event.RESPONSE, blpapi.Event.PARTIAL_RESPONSE):
                self.process_response(event)
            else:
                self.process_misc_events(event)
        except Exception as e:
            print(f"处理事件时出错: {str(e)}", file=sys.stderr)
        return False

    def process_session_status(self, event, session):
        """处理会话状态事件"""
        print("处理 SESSION_STATUS 事件")
        for msg in event:
            if msg.messageType() == SESSION_STARTED:
                print("会话已启动，正在打开历史数据服务...")
                session.openServiceAsync(HISTORY_SERVICE)
            elif msg.messageType() == SESSION_STARTUP_FAILURE:
                print("错误：会话启动失败", file=sys.stderr)
                global b_end
                b_end = True

    def process_service_status(self, event, session):
        """处理服务状态事件（核心参数修复）"""
        print("处理 SERVICE_STATUS 事件")
        for msg in event:
            if msg.messageType() == SERVICE_OPENED:
                print("历史数据服务已打开，正在构建 GetFills 请求...")
                service = session.getService(HISTORY_SERVICE)
                request = service.createRequest("GetFills")

                # 1. 修复时区问题：扩大时间范围（覆盖UTC+/-12小时，确保包含本地时区记录）
                # 原范围：2025-11-01 UTC → 调整为2025-10-31 UTC 至 2025-12-01 UTC
                request.set("FromDateTime", "2025-10-31T00:00:00.000+00:00")
                request.set("ToDateTime", "2025-12-01T23:59:59.999+00:00")

                # 2. 修复UUID类型：同时尝试整数+字符串格式（关键！）
                scope = request.getElement("Scope")
                # 方案A：按Uuids筛选（同时传数字+字符串格式，兼容不同环境）
                scope.setChoice("Uuids")
                uuids_elem = scope.getElement("Uuids")
                uuids_elem.appendValue(30937014)  # 整数格式
                uuids_elem.appendValue("30937014") # 字符串格式（部分环境要求）

                # 方案B：若Uuids仍无数据，取消注释改用Team筛选（联系管理员获取该UUID所属Team）
                # scope.setChoice("Team")
                # scope.setElement("Team", "YOUR_TEAM_NAME")  # 替换为UUID 30937014所属团队

                # 方案C：改用TradingSystem筛选（若适用）
                # scope.setChoice("TradingSystem")
                # scope.setElement("TradingSystem", "YOUR_TRADING_SYSTEM")

                # 3. 补充可选参数：添加EMSX_SESSION_ID（部分环境必填）
                # 联系管理员获取该UUID的SESSION ID，若无则注释
                # request.set("EMSX_SESSION_ID", "YOUR_EMSX_SESSION_ID")

                print(f"发送请求: {request.toString()}")
                self.request_id = blpapi.CorrelationId()
                print(f"请求关联ID: {self.request_id.value()}")
                session.sendRequest(request, correlationId=self.request_id)

            elif msg.messageType() == SERVICE_OPEN_FAILURE:
                print("错误：历史数据服务打开失败", file=sys.stderr)
                global b_end
                b_end = True

    def process_response(self, event):
        """处理响应事件（增强日志）"""
        print("处理 RESPONSE 事件")
        global b_end
        for msg in event:
            resp_cid = msg.correlationIds()[0].value() if msg.correlationIds() else "无"
            msg_type = msg.messageType()
            print(f"响应消息类型: {msg_type} | 响应关联ID: {resp_cid}")
            
            try:
                if msg_type == ERROR_INFO:
                    error_code = msg.getElementAsInteger("ERROR_CODE")
                    error_msg = msg.getElementAsString("ERROR_MESSAGE")
                    print(f"❌ 错误代码: {error_code}，错误信息: {error_msg}")
                    # 常见错误码参考：
                    # 0x0006000d：操作/服务不存在 → 切换服务地址
                    # 0x0006001e：权限不足 → 联系管理员开通权限
                elif msg_type == GET_FILLS_RESPONSE:
                    fills = msg.getElement("Fills")
                    fill_count = fills.numValues()
                    print(f"✅ 共获取到 {fill_count} 条填充记录：")
                    if fill_count > 0:
                        for idx, fill in enumerate(fills.values()):
                            # 提取更多字段，确认数据完整性
                            fill_id = fill.getElement("FillId").getValueAsInteger()
                            order_id = fill.getElement("OrderId").getValueAsInteger()
                            fill_time = fill.getElement("DateTimeOfFill").getValueAsString()
                            fill_price = fill.getElement("FillPrice").getValueAsFloat()
                            fill_shares = fill.getElement("FillShares").getValueAsFloat()
                            side = fill.getElement("Side").getValueAsString()
                            # 新增字段：确认UUID/Team信息
                            user_uuid = fill.getElement("UserUUID").getValueAsString() if fill.hasElement("UserUUID") else "未知"
                            team = fill.getElement("Team").getValueAsString() if fill.hasElement("Team") else "未知"

                            print(f"\n记录{idx+1}:")
                            print(f"  填充ID: {fill_id} | 订单ID: {order_id} | 用户UUID: {user_uuid} | 团队: {team}")
                            print(f"  时间: {fill_time} | 价格: {fill_price} | 数量: {fill_shares} | 方向: {side}")
                    else:
                        print("⚠️  仍无数据，建议：")
                        print("  1. 确认该UUID的交易记录时间是否在 2025-10-31 至 2025-12-01 UTC 范围内")
                        print("  2. 切换Scope为Team/TradingSystem（联系管理员获取所属团队）")
                        print("  3. 验证账户是否有该UUID的历史数据查询权限")
            except Exception as e:
                print(f"解析响应时出错: {str(e)}")
            
            b_end = True

    def process_misc_events(self, event):
        """处理其他未定义类型的事件"""
        print(f"处理未定义事件: {event.eventType()}")
        for msg in event:
            print(f"事件内容: {msg.toString()}")


def main():
    # 配置会话选项（新增：启用详细日志，排查连接问题）
    session_options = blpapi.SessionOptions()
    session_options.setServerHost(HOST)
    session_options.setServerPort(PORT)
    # 启用Bloomberg API详细日志（生成日志文件，方便排查）
    session_options.setLogLevel(blpapi.LogLevel.TRACE)
    session_options.setLogFileName("emsx_history_log.txt")

    print(f"连接到 Bloomberg 服务: {HOST}:{PORT}")
    print(f"日志文件已生成：emsx_history_log.txt（可查看详细请求/响应）")

    event_handler = SessionEventHandler()
    session = blpapi.Session(session_options, event_handler.processEvent)

    if not session.startAsync():
        print("会话启动失败", file=sys.stderr)
        return

    # 超时保护（延长至60秒，避免响应慢导致超时）
    timeout = 60
    start_time = time.time()
    global b_end
    while not b_end and (time.time() - start_time) < timeout:
        time.sleep(0.1)
    
    if not b_end:
        print(f"⚠️  警告：请求超时（{timeout}秒），未收到响应")
        b_end = True

    session.stop()
    print("会话已关闭")


if __name__ == "__main__":
    print("Bloomberg EMSX API 历史填充数据查询示例（最终修复版）")
    print(f"查询UUID: 30937014 | 时间范围: 2025-10-31 UTC 至 2025-12-01 UTC（覆盖时区偏差）")
    try:
        main()
    except KeyboardInterrupt:
        print("用户中断操作，程序停止")

__copyright__ = """
Copyright 2025. Bloomberg Finance L.P.
保留所有权利。
"""

EMSX History Client
Successfully connected to Bloomberg EMSX API

1. Getting historical data for date range...
Sending request: FromDateTime=2025-12-01T00:00:00.000+00:00, ToDateTime=2025-12-05T23:59:00.000+00:00, UUID=30937014
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response
No fill records found in response


In [ ]:
#!/usr/bin/env python3
"""
EMSXHistory.py - Example script to retrieve historical EMSX data
Compatible with Bloomberg EMSX API
"""

import blpapi
import datetime
import pandas as pd
from typing import Optional, Dict, List, Any


class EMSXHistoryClient:
    """
    Client for retrieving historical EMSX data from Bloomberg
    """
    
    def __init__(self, host: str = 'localhost', port: int = 8194):
        """
        Initialize EMSX history client
        
        Args:
            host: Bloomberg API host (default: localhost)
            port: Bloomberg API port (default: 8194)
        """
        self.host = host
        self.port = port
        self.session = None
        self.service = None
        
    def connect(self) -> bool:
        """
        Establish connection to Bloomberg API
        
        Returns:
            bool: True if connection successful, False otherwise
        """
        try:
            # Create session options
            session_options = blpapi.SessionOptions()
            session_options.setServerHost(self.host)
            session_options.setServerPort(self.port)
            
            # Create and start session
            self.session = blpapi.Session(session_options)
            
            if not self.session.start():
                print("Failed to start session")
                return False
                
            if not self.session.openService("//blp/emapisvc"):
                print("Failed to open service")
                return False
                
            self.service = self.session.getService("//blp/emapisvc")
            print("Successfully connected to Bloomberg EMSX API")
            return True
            
        except Exception as e:
            print(f"Connection error: {e}")
            return False
    
    def get_history(self,
                    start_datetime: str,
                    end_datetime: str,
                    uuid: str) -> pd.DataFrame:
        """
        Retrieve historical EMSX FILL data.

        Args:
            start_datetime: Start date/time in ISO format (e.g., '2025-12-01T00:00:00.000+00:00').
            end_datetime: End date/time in ISO format (e.g., '2025-12-05T23:59:00.000+00:00').
            uuid: Bloomberg UUID of the trader (STRING format, e.g., "30937014").

        Returns:
            pd.DataFrame: Historical fill data.
        """
        if not self.session:
            raise ConnectionError("Not connected to Bloomberg API")

        # 选择正确的服务路径（生产/测试环境）
        HISTORY_SERVICE = "//blp/emsx.history"  # 测试环境使用 "//blp/emsx.history.uat"
        
        if not self.session.openService(HISTORY_SERVICE):
            print(f"Failed to open history service: {HISTORY_SERVICE}")
            return pd.DataFrame()
        
        service = self.session.getService(HISTORY_SERVICE)

        try:
            # 创建GetFills请求
            request = service.createRequest("GetFills")
            request.set("FromDateTime", start_datetime)
            request.set("ToDateTime", end_datetime)

            # 正确设置Scope（UUID必须是字符串类型）
            scope = request.getElement("Scope")
            scope.setChoice("Uuids")
            uuid_elem = scope.getElement("Uuids")
            uuid_elem.appendValue(str(uuid))  # 强制转换为字符串

            print(f"Sending request: FromDateTime={start_datetime}, ToDateTime={end_datetime}, UUID={uuid}")
            
            # 发送请求
            correlation_id = blpapi.CorrelationId(1)
            self.session.sendRequest(request, correlationId=correlation_id)
            data_records = []
            
            # 循环控制变量（防止无限循环）
            max_attempts = 100  # 最大尝试次数
            attempts = 0
            response_received = False

            # 处理响应事件
            while attempts < max_attempts:
                attempts += 1
                event = self.session.nextEvent(3000)  # 3秒超时
                
                # 处理响应事件
                if event.eventType() in (blpapi.Event.RESPONSE, blpapi.Event.PARTIAL_RESPONSE):
                    response_received = True
                    for msg in event:
                        # 处理错误信息
                        if msg.messageType() == "ErrorInfo":
                            error_code = msg.getElementAsInteger("error_code")
                            error_msg = msg.getElementAsString("error_message")
                            print(f"API Error: {error_code} - {error_msg}")
                            continue
                        
                        # 处理GetFills响应
                        if msg.messageType() == "GetFillsResponse":
                            # 详细日志帮助排查数据为空的问题
                            print(f"Response received - Message numElements: {msg.numElements()}")
                            
                            # 检查fill数据（兼容不同的字段名称）
                            fill_numElements = ["fill", "Fills", "fills"]
                            fill_found = False
                            
                            for elem_name in fill_numElements:
                                if msg.hasElement(elem_name):
                                    fills = msg.getElement(elem_name)
                                    if fills.numValues() > 0:
                                        fill_found = True
                                        print(f"Found {fills.numValues()} fill records in '{elem_name}'")
                                        
                                        for i in range(fills.numValues()):
                                            fill = fills.getValueAsElement(i)
                                            record = {}
                                            # 提取字段（增加容错处理）
                                            fields = [
                                                ("DateTimeOfFill", str),
                                                ("Ticker", str),
                                                ("FillShares", int),
                                                ("FillPrice", float),
                                                ("Broker", str),
                                                ("OrderId", int),
                                                ("Side", str),
                                                ("Account", str),
                                                ("Strategy", str),
                                                ("Trader", str)
                                            ]
                                            
                                            for field_name, field_type in fields:
                                                try:
                                                    if field_type == str:
                                                        record[field_name] = fill.getElementAsString(field_name)
                                                    elif field_type == int:
                                                        record[field_name] = fill.getElementAsInteger(field_name)
                                                    elif field_type == float:
                                                        record[field_name] = fill.getElementAsFloat(field_name)
                                                except blpapi.exception.NotFoundException:
                                                    record[field_name] = None
                                            
                                            data_records.append(record)
                                    else:
                                        print(f"'{elem_name}' element exists but has no values")
                                    break
                            
                            if not fill_found:
                                print("No fill records found in response - Detailed debug:")
                                print(f"All message numElements: {[e.name() for e in msg.numElements()]}")
                                # 检查是否有权限问题
                                if msg.hasElement("error"):
                                    print(f"Error details: {msg.getElement('error')}")
                
                # 处理请求状态（退出循环）
                elif event.eventType() == blpapi.Event.REQUEST_STATUS:
                    print("Request status received - ending loop")
                    break
                
                # 处理超时事件（仅打印，不退出）
                elif event.eventType() == blpapi.Event.TIMEOUT:
                    print(f"Timeout ({attempts}/{max_attempts}) - no event received")
                    continue
                
                # 如果收到完整响应，退出循环
                if response_received and event.eventType() == blpapi.Event.RESPONSE:
                    print("Complete response received - ending loop")
                    break

            # 循环结束后检查
            if attempts >= max_attempts:
                print(f"Warning: Reached maximum attempts ({max_attempts}) - loop terminated")
            
            return pd.DataFrame(data_records)

        except blpapi.exception.NotFoundException as e:
            print(f"Request failed (missing field): {e}")
            return pd.DataFrame()
        except Exception as e:
            print(f"Unexpected error: {e}")
            import traceback
            traceback.print_exc()
            return pd.DataFrame()

    def get_history_by_order_id(self, 
                               order_id: int,
                               days_back: int = 30,
                               uuid: str = "30937014") -> pd.DataFrame:
        """
        Get history for a specific order ID
        
        Args:
            order_id: EMSX order ID
            days_back: Number of days to look back
            uuid: Bloomberg UUID (string format)
            
        Returns:
            pd.DataFrame: Historical data for the order
        """
        end_date = datetime.datetime.now(datetime.timezone.utc)
        start_date = end_date - datetime.timedelta(days=days_back)
        
        # 确保日期格式正确（UTC时区）
        start_str = start_date.strftime("%Y-%m-%dT%H:%M:%S.000+00:00")
        end_str = end_date.strftime("%Y-%m-%dT%H:%M:%S.000+00:00")
        
        print(f"Getting history from {start_str} to {end_str} for order {order_id}")
        
        df = self.get_history(start_str, end_str, uuid)
        
        if not df.empty and "OrderId" in df.columns:
            df = df[df["OrderId"] == order_id]
        
        return df
    
    def disconnect(self):
        """Disconnect from Bloomberg API"""
        if self.session:
            self.session.stop()
            print("Disconnected from Bloomberg EMSX API")


def main():
    """Example usage of EMSXHistoryClient"""
    
    # Create client
    client = EMSXHistoryClient()
    
    try:
        # Connect to Bloomberg
        if not client.connect():
            print("Failed to connect. Make sure Bloomberg Terminal is running.")
            return
        
        # Example 1: Get history for date range
        print("\n1. Getting historical data for date range...")
        # 使用UTC时区的正确格式
        start_datetime = "2025-12-01T00:00:00.000+00:00"
        end_datetime = "2025-12-05T23:59:59.999+00:00"
        
        # UUID必须是字符串格式！
        uuid = "30937014"  # 替换为你的实际UUID（字符串类型）
        
        history_data = client.get_history(
            start_datetime=start_datetime,
            end_datetime=end_datetime,
            uuid=uuid   
        )
        
        if not history_data.empty:
            print(f"Retrieved {len(history_data)} records")
            print("\nFirst 5 records:")
            print(history_data.head())
            
            # Save to CSV
            history_data.to_csv("emsx_history.csv", index=False)
            print("\nData saved to emsx_history.csv")
            
            # Display column information
            print(f"\nColumns available: {list(history_data.columns)}")
        else:
            print("\n=== No data retrieved. Troubleshooting checklist ===")
            print("1. Verify UUID is correct (must be string format, not integer)")
            print("2. Check if there are actual fills in the specified date range")
            print("3. Confirm Bloomberg Terminal has EMSX history access permissions")
            print("4. For test environment, use HISTORY_SERVICE = '//blp/emsx.history.uat'")
            print("5. Ensure date range uses UTC timezone (+00:00)")
            print("6. Check if the UUID has access to the fill data (permissions)")
            print("7. Verify Bloomberg API version is compatible (upgrade if needed)")
        
    except Exception as e:
        print(f"Error in main: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        # Always disconnect
        client.disconnect()


def test_connection():
    """Simple test to check if we can connect and make a basic request"""
    client = EMSXHistoryClient()
    
    try:
        if client.connect():
            print("✓ Successfully connected to Bloomberg")
            
            # Test with very recent date range (UTC timezone)
            end_time = datetime.datetime.now(datetime.timezone.utc)
            start_time = end_time - datetime.timedelta(days=7)  # 扩大测试范围到7天
            
            start_str = start_time.strftime("%Y-%m-%dT%H:%M:%S.000+00:00")
            end_str = end_time.strftime("%Y-%m-%dT%H:%M:%S.000+00:00")
            
            print(f"\nTesting with date range: {start_str} to {end_str}")
            
            # UUID必须是字符串！
            test_data = client.get_history(
                start_datetime=start_str,
                end_datetime=end_str,
                uuid="30937014"  # 替换为实际UUID
            )
            
            if not test_data.empty:
                print(f"✓ Successfully retrieved {len(test_data)} records")
            else:
                print("⚠ No data returned - check troubleshooting checklist")
    finally:
        client.disconnect()


if __name__ == "__main__":
    print("EMSX History Client")
    print("=" * 50)
    
    # Run basic example
    main()
    
    # Uncomment to run connection test
    # print("\n" + "=" * 50)
    # print("Running connection test...")
    # test_connection()

EMSX History Client
Successfully connected to Bloomberg EMSX API

1. Getting historical data for date range...
Sending request: FromDateTime=2025-12-01T00:00:00.000+00:00, ToDateTime=2025-12-05T23:59:59.999+00:00, UUID=30937014
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 'Fills'
Response received - Message numElements: 1
Found 1000 fill records in 